# 03 — Drift Detection & Bandit

This notebook covers **Gap 3**: drift that is detected AND acted on automatically, plus automatic traffic shifting via bandits.

Topics covered:
- `DriftDetector` — PSI and KS test explained
- `DriftReport` — reading every field
- PSI severity levels: none / moderate / severe
- `detector.watch()` — monitoring a live prediction stream
- `on_drift(action="rollback")` — automatic rollback
- `on_drift(action=callable)` — custom callback
- `DriftReport.export()` — the HTML drift report
- `Bandit` — Thompson sampling explained step by step
- `Bandit` — epsilon-greedy explained
- `bandit.allocations()` — watching traffic shift over time
- `bandit.summary()` — reading P(challenger > baseline)
- Comparing fixed split vs bandit on the same data

In [1]:
import numpy as np
from evalbridge import Experiment, DriftDetector
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=2000, n_features=20, n_informative=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

baseline_model   = LogisticRegression(random_state=42, max_iter=1000).fit(X_train, y_train)
challenger_model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

base_preds = baseline_model.predict_proba(X_test)[:, 1]
chal_preds = challenger_model.predict_proba(X_test)[:, 1]

print("Setup complete")

Setup complete


## 1. DriftDetector — the two tests

**PSI (Population Stability Index):**
Bins predictions into 10 buckets from 0 to 1. Measures how much the
bucket proportions shifted. Formula: `Σ (cur% − ref%) × log(cur% / ref%)`.

**KS test (Kolmogorov-Smirnov):**
Measures the maximum vertical gap between the two cumulative distribution
functions. A small p-value means the distributions are significantly different.

In [2]:
rng = np.random.default_rng(0)

# Create three scenarios to compare
stable      = base_preds + rng.normal(0, 0.005, len(base_preds))  # tiny noise
stable      = np.clip(stable, 0, 1)

moderate    = rng.beta(1.5, 1.5, len(base_preds))                 # slightly shifted

severe      = rng.beta(0.3, 5, len(base_preds))                   # heavily shifted toward 0

detector = DriftDetector(reference=base_preds, threshold=0.2)

print(f"{'Scenario':<12} {'PSI':>8} {'KS stat':>10} {'KS p-val':>12} {'Severity':<12} {'Alert'}")
print("-" * 65)
for name, arr in [("stable", stable), ("moderate", moderate), ("severe", severe)]:
    r = detector.check(arr)
    print(f"{name:<12} {r.psi:>8.4f} {r.ks_stat:>10.4f} {r.ks_pvalue:>12.4f} {r.severity:<12} {r.alert}")

Scenario          PSI    KS stat     KS p-val Severity     Alert
-----------------------------------------------------------------
stable         0.0004     0.0225       0.9875 none         False
moderate       0.9110     0.2275       0.0000 severe       True
severe         9.2668     0.5887       0.0000 severe       True


## 2. Reading every field of DriftReport

In [3]:
report = detector.check(severe)

print(f"psi        : {report.psi:.4f}")
print(f"             < 0.10  → none")
print(f"             0.10-0.20 → moderate")
print(f"             > 0.20  → severe  ← we are here")
print()
print(f"ks_stat    : {report.ks_stat:.4f}  (max gap between CDFs, 0=identical)")
print(f"ks_pvalue  : {report.ks_pvalue:.6f}  (< 0.05 means significantly different)")
print(f"alert      : {report.alert}    (PSI > threshold={report.threshold})")
print(f"severity   : {report.severity}")
print(f"threshold  : {report.threshold}")
print(f"summary    : {report.summary}")

psi        : 9.2668
             < 0.10  → none
             0.10-0.20 → moderate
             > 0.20  → severe  ← we are here

ks_stat    : 0.5887  (max gap between CDFs, 0=identical)
ks_pvalue  : 0.000000  (< 0.05 means significantly different)
alert      : True    (PSI > threshold=0.2)
severity   : severe
threshold  : 0.2
summary    : Severe drift detected (PSI=9.2668) — investigate


## 3. PSI threshold control

In [4]:
# Same data, different thresholds
print("Effect of threshold on alert for 'moderate' drift:")
print(f"{'threshold':>12} {'PSI':>8} {'alert'}")
print("-" * 30)
for thresh in [0.05, 0.10, 0.15, 0.20, 0.30]:
    d = DriftDetector(reference=base_preds, threshold=thresh)
    r = d.check(moderate)
    print(f"{thresh:>12.2f} {r.psi:>8.4f}  {r.alert}")

Effect of threshold on alert for 'moderate' drift:
   threshold      PSI alert
------------------------------
        0.05   0.9110  True
        0.10   0.9110  True
        0.15   0.9110  True
        0.20   0.9110  True
        0.30   0.9110  True


## 4. watch() — streaming production monitoring

In [5]:
# Simulate a production stream: starts stable, then drifts
stable_stream  = rng.uniform(0.3, 0.7, 400).tolist()  # first 400: normal
drifted_stream = rng.beta(0.2, 5, 400).tolist()       # next 400: drifted
full_stream    = stable_stream + drifted_stream

detector_watch = DriftDetector(reference=base_preds, threshold=0.15)

alerts = []

def on_drift_callback(report):
    alerts.append(report)
    print(f"  DRIFT DETECTED: PSI={report.psi:.4f}  severity={report.severity}")

print("Monitoring stream (interval=100):")
detector_watch.watch(full_stream, interval=100, on_drift=on_drift_callback)

print(f"\nTotal alerts fired: {len(alerts)}")

Monitoring stream (interval=100):
  DRIFT DETECTED: PSI=18.1255  severity=severe
  DRIFT DETECTED: PSI=18.1179  severity=severe
  DRIFT DETECTED: PSI=18.1553  severity=severe
  DRIFT DETECTED: PSI=18.1563  severity=severe
  DRIFT DETECTED: PSI=13.1348  severity=severe
  DRIFT DETECTED: PSI=13.1330  severity=severe
  DRIFT DETECTED: PSI=9.8779  severity=severe
  DRIFT DETECTED: PSI=10.1602  severity=severe

Total alerts fired: 8


## 5. on_drift — automatic rollback when drift is detected

In [6]:
rollback_log = []
alert_log    = []

exp_drift = Experiment("drift_demo", min_samples=50, min_confidence=0.7)
exp_drift.log("baseline",   y_true=y_test.tolist(), y_pred=base_preds.tolist())
exp_drift.log("challenger", y_true=y_test.tolist(), y_pred=chal_preds.tolist())

# Register two callbacks on the same detector
exp_drift.on_drift(threshold=0.05, action="rollback")                            # built-in string
exp_drift.on_drift(threshold=0.05, action=lambda r: rollback_log.append(True))  # custom callable
exp_drift.on_confidence(threshold=0.5, action=lambda r: alert_log.append(r.winner))  # confidence hook

# Inject drifted challenger data to trigger the drift detector
drifted_preds = rng.beta(0.2, 8, 300).tolist()
exp_drift.log("challenger", y_true=[0] * 300, y_pred=drifted_preds)

result_drift = exp_drift.evaluate()

print(f"rollback fired    : {len(rollback_log) > 0}")
print(f"custom callback   : {len(rollback_log)} times")
print(f"confidence callback: winner = {alert_log[-1] if alert_log else 'not fired'}")

[evalbridge] rollback() called — reverting to baseline
rollback fired    : True
custom callback   : 1 times
confidence callback: winner = challenger


## 6. DriftReport HTML export

In [7]:
drift_report = detector.check(severe)
drift_report.export("/tmp/drift_report.html")
print("Drift report written to /tmp/drift_report.html")
print(f"Summary: {drift_report.summary}")

[evalbridge] Drift report saved to /tmp/drift_report.html
Drift report written to /tmp/drift_report.html
Summary: Severe drift detected (PSI=9.2668) — investigate


## 7. Thompson Sampling Bandit — step by step

Thompson sampling uses a Beta distribution for each arm.
- Each **correct** prediction increments `alpha` (the "win" count)
- Each **incorrect** prediction increments `beta` (the "loss" count)
- On each `route()` call, one sample is drawn from each arm's Beta distribution
- The arm with the higher sample wins the request

As the better arm accumulates wins, its Beta distribution shifts higher → it wins more samples → it gets more traffic.

In [8]:
exp_bandit = Experiment("bandit_demo", min_samples=10, method="bandit")
exp_bandit.log("baseline",   y_true=y_test[:100].tolist(), y_pred=base_preds[:100].tolist())
exp_bandit.log("challenger", y_true=y_test[:100].tolist(), y_pred=chal_preds[:100].tolist())

bandit = exp_bandit.as_bandit(strategy="thompson")

# Show starting state
print("Starting Beta parameters:")
print(f"  baseline   alpha={bandit._alpha['baseline']}  beta={bandit._beta['baseline']}")
print(f"  challenger alpha={bandit._alpha['challenger']}  beta={bandit._beta['challenger']}")
print(f"Starting allocations: {bandit.allocations()}")

Starting Beta parameters:
  baseline   alpha=1  beta=1
  challenger alpha=1  beta=1
Starting allocations: {'baseline': 0.5, 'challenger': 0.5}


In [9]:
# Simulate rounds: challenger wins 80% of the time, baseline wins 65%
rng_b = np.random.default_rng(42)

snapshots = []

for round_num in range(1, 501):
    arm = bandit.route()
    reward = float(rng_b.random() < (0.80 if arm == "challenger" else 0.65))
    bandit.update(arm, reward)

    if round_num in (1, 10, 50, 100, 200, 500):
        allocs = bandit.allocations()
        snapshots.append((round_num, allocs['challenger']))

print(f"{'Round':>8} {'Challenger %':>14}")
print("-" * 25)
for rnd, chal_alloc in snapshots:
    bar = "█" * int(chal_alloc * 30)
    print(f"{rnd:>8}  {chal_alloc:>6.1%}  {bar}")

print()
bandit.summary()

   Round   Challenger %
-------------------------
       1    0.0%  
      10   30.0%  █████████
      50   80.0%  ████████████████████████
     100   90.0%  ███████████████████████████
     200   95.0%  ████████████████████████████
     500   98.0%  █████████████████████████████

Strategy: thompson
  baseline   :   2.0%
  challenger :  98.0%
  P(challenger > baseline): 0.999


'Strategy: thompson\n  baseline   :   2.0%\n  challenger :  98.0%\n  P(challenger > baseline): 0.999'

## 8. Epsilon-Greedy Bandit

In [10]:
exp_eg = Experiment("eg_demo", min_samples=10, method="bandit")
exp_eg.log("baseline",   y_true=y_test[:100].tolist(), y_pred=base_preds[:100].tolist())
exp_eg.log("challenger", y_true=y_test[:100].tolist(), y_pred=chal_preds[:100].tolist())

bandit_eg = exp_eg.as_bandit(strategy="epsilon_greedy", epsilon=0.15)

rng_eg = np.random.default_rng(0)
routes = {"baseline": 0, "challenger": 0}
explore_count = 0

for _ in range(500):
    arm = bandit_eg.route()
    reward = float(rng_eg.random() < (0.80 if arm == "challenger" else 0.65))
    bandit_eg.update(arm, reward)
    routes[arm] += 1

print("Epsilon-greedy (epsilon=0.15) after 500 rounds:")
print(f"  baseline   : {routes['baseline']} routes ({routes['baseline']/500:.1%})")
print(f"  challenger : {routes['challenger']} routes ({routes['challenger']/500:.1%})")
bandit_eg.summary()

Epsilon-greedy (epsilon=0.15) after 500 rounds:
  baseline   : 68 routes (13.6%)
  challenger : 432 routes (86.4%)
Strategy: epsilon_greedy
  baseline   :  13.6%
  challenger :  86.4%
  P(challenger > baseline): 1.000


'Strategy: epsilon_greedy\n  baseline   :  13.6%\n  challenger :  86.4%\n  P(challenger > baseline): 1.000'

## 9. Thompson vs Epsilon-Greedy — side by side comparison

In [11]:
def run_bandit(strategy, challenger_win_rate=0.80, baseline_win_rate=0.65, rounds=500, seed=42):
    """Run a bandit simulation and return challenger allocation history."""
    exp_sim = Experiment("sim", min_samples=5, method="bandit")
    exp_sim.log("baseline",   y_true=[1]*50, y_pred=[0.6]*50)
    exp_sim.log("challenger", y_true=[1]*50, y_pred=[0.7]*50)

    b = exp_sim.as_bandit(strategy=strategy)
    rng_sim = np.random.default_rng(seed)
    history = []

    for i in range(1, rounds + 1):
        arm = b.route()
        wr = challenger_win_rate if arm == "challenger" else baseline_win_rate
        b.update(arm, float(rng_sim.random() < wr))
        if i % 25 == 0:
            history.append((i, b.allocations()['challenger']))

    return history

thompson_hist = run_bandit("thompson")
eg_hist       = run_bandit("epsilon_greedy")

print(f"{'Round':>8} {'Thompson %':>12} {'Epsilon-Greedy %':>18}")
print("-" * 42)
for (r1, t_alloc), (r2, eg_alloc) in zip(thompson_hist, eg_hist):
    print(f"{r1:>8} {t_alloc:>12.1%} {eg_alloc:>18.1%}")

print("\nThompson tends to converge faster and allocate more traffic")
print("to the winner. Epsilon-greedy keeps exploring at a fixed rate.")

   Round   Thompson %   Epsilon-Greedy %
------------------------------------------
      25        96.0%              84.0%
      50        98.0%              90.0%
      75        97.3%              92.0%
     100        96.0%              92.0%
     125        95.2%              92.8%
     150        96.0%              94.0%
     175        94.9%              93.1%
     200        94.5%              94.0%
     225        95.1%              94.7%
     250        95.6%              95.2%
     275        95.6%              94.9%
     300        95.7%              95.0%
     325        96.0%              95.4%
     350        96.3%              94.9%
     375        96.5%              94.9%
     400        96.8%              94.5%
     425        96.5%              94.8%
     450        96.7%              94.4%
     475        96.6%              94.5%
     500        96.8%              94.4%

Thompson tends to converge faster and allocate more traffic
to the winner. Epsilon-greedy keeps

## 10. Fixed split vs Bandit — cumulative reward comparison

In [12]:
# Which approach earns more total reward over 500 requests?

ROUNDS = 500
CHAL_WIN_RATE = 0.80
BASE_WIN_RATE = 0.65
rng_c = np.random.default_rng(99)

# Fixed 10% split — always 90% baseline, 10% challenger
fixed_reward = 0
for _ in range(ROUNDS):
    arm = "challenger" if rng_c.random() < 0.10 else "baseline"
    wr = CHAL_WIN_RATE if arm == "challenger" else BASE_WIN_RATE
    fixed_reward += int(rng_c.random() < wr)

# Thompson bandit — shifts traffic toward better arm
rng_c2 = np.random.default_rng(99)
exp_c = Experiment("reward_sim", min_samples=5, method="bandit")
exp_c.log("baseline",   y_true=[1]*10, y_pred=[0.6]*10)
exp_c.log("challenger", y_true=[1]*10, y_pred=[0.7]*10)
bandit_c = exp_c.as_bandit("thompson")

bandit_reward = 0
for _ in range(ROUNDS):
    arm = bandit_c.route()
    wr = CHAL_WIN_RATE if arm == "challenger" else BASE_WIN_RATE
    reward = int(rng_c2.random() < wr)
    bandit_c.update(arm, float(reward))
    bandit_reward += reward

print(f"Over {ROUNDS} requests (challenger={CHAL_WIN_RATE:.0%}, baseline={BASE_WIN_RATE:.0%}):")
print(f"  Fixed 10% split  : {fixed_reward} successes ({fixed_reward/ROUNDS:.1%})")
print(f"  Thompson bandit  : {bandit_reward} successes ({bandit_reward/ROUNDS:.1%})")
print(f"  Bandit advantage : +{bandit_reward - fixed_reward} successes")
print()
print(f"Final bandit allocations: {bandit_c.allocations()}")

Over 500 requests (challenger=80%, baseline=65%):
  Fixed 10% split  : 327 successes (65.4%)
  Thompson bandit  : 391 successes (78.2%)
  Bandit advantage : +64 successes

Final bandit allocations: {'baseline': 0.068, 'challenger': 0.932}
